# Analisis Sentimen & Identifikasi Penyebab Keluhan Pelanggan Tokopedia

**Dataset:** Tokopedia Product Reviews 2025 (Kaggle - salmanabdu/tokopedia-product-reviews-2025)

**Tujuan Proyek:**
1. Membangun model klasifikasi sentimen dari teks review (tanpa mengandalkan rating)
2. Menemukan topik/tema keluhan utama dari review negatif (topic modeling)
3. Menghubungkan topik keluhan dengan kategori produk untuk insight bisnis

**Catatan penting:** Kolom `sentiment_label` pada dataset ini merupakan turunan langsung dari `rating` (rating 4-5 = positive, 3 = neutral, 1-2 = negative), bukan hasil analisis teks. Hal ini telah diverifikasi pada tahap EDA.

## 1. Setup & Load Dataset

In [ ]:
# Install dependencies
!pip install kagglehub[pandas-datasets] --quiet

In [ ]:
import kagglehub
kagglehub.login()  # masukkan API token Kaggle yang baru saat diminta

In [ ]:
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

file_path = "tokopedia_product_reviews_2025.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "salmanabdu/tokopedia-product-reviews-2025",
    file_path,
)

print("Jumlah baris, kolom:", df.shape)
df.head()

## 2. Exploratory Data Analysis (EDA)

Tahap ini untuk memahami struktur data sebelum diproses lebih lanjut: missing value, distribusi kategori, distribusi rating, dan distribusi sentimen.

In [ ]:
# Info umum & missing value
df.info()
print("\nMissing value per kolom:")
print(df.isnull().sum())

In [ ]:
# Distribusi kategori produk
print(df['product_category'].value_counts())

In [ ]:
# Distribusi rating dan sentiment_label
print("Distribusi rating:")
print(df['rating'].value_counts().sort_index())

print("\nDistribusi sentiment_label:")
print(df['sentiment_label'].value_counts())
print(df['sentiment_label'].value_counts(normalize=True) * 100)

In [ ]:
# Verifikasi: sentiment_label adalah turunan langsung dari rating
df.groupby('sentiment_label')['rating'].agg(['mean', 'count', 'min', 'max'])

**Temuan EDA:**
- Data bersih, tanpa missing value pada kolom-kolom penting (kecuali `product_variant` yang memang kosong untuk produk dengan satu varian).
- Dataset sangat *imbalanced*: ~97.6% review berlabel `positive`, sisanya `negative` (~1.2%) dan `neutral` (~1.2%).
- `sentiment_label` terbukti murni turunan dari `rating` (rating 1-2 -> negative, 3 -> neutral, 4-5 -> positive), sehingga valid digunakan sebagai target klasifikasi, namun tidak mengandung informasi tambahan dari teks.

## 3. Text Cleaning

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)       # hapus URL
    text = re.sub(r'[^a-z\s]', '', text)      # hapus angka & tanda baca
    text = re.sub(r'\s+', ' ', text).strip()  # rapikan spasi berlebih
    return text

df['clean_review'] = df['review_text'].apply(clean_text)
df[['review_text', 'clean_review']].head()

## 4. Klasifikasi Sentimen dari Teks Review

Membangun model klasifikasi sentimen sendiri (TF-IDF + Naive Bayes / SVM) yang dilatih langsung dari data review Tokopedia, alih-alih menggunakan model pretrained umum (yang terbukti kurang akurat untuk teks informal/gaul pada uji coba awal).

Karena data sangat *imbalanced*, digunakan `stratify` saat split data dan `class_weight='balanced'` pada model agar model tidak hanya menebak kelas mayoritas (`positive`).

In [ ]:
X_text = df['clean_review']
y = df['sentiment_label']

In [ ]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print("Jumlah data training:", len(X_train_text))
print("Jumlah data testing:", len(X_test_text))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("Bentuk data training setelah TF-IDF:", X_train_tfidf.shape)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Model 1: Naive Bayes (baseline)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)

# Model 2: LinearSVC dengan class_weight='balanced' agar lebih peka pada kelas minoritas
svm_model = LinearSVC(class_weight='balanced', max_iter=2000)
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)

In [ ]:
print("=== Naive Bayes ===")
print("Akurasi:", accuracy_score(y_test, nb_pred))
print(classification_report(y_test, nb_pred))

print("\n=== SVM (LinearSVC) ===")
print("Akurasi:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred))

**Temuan:** Naive Bayes memiliki akurasi keseluruhan lebih tinggi, namun gagal mengenali kelas minoritas (`negative`/`neutral`) karena kecenderungan menebak kelas mayoritas. SVM dengan `class_weight='balanced'` memiliki akurasi lebih rendah namun performa (F1-score) jauh lebih baik pada kelas `negative` dan `neutral`, sehingga dipilih sebagai model final untuk tujuan mendeteksi keluhan pelanggan.

## 5. Topic Modeling pada Review Negatif

Menggunakan NMF (Non-negative Matrix Factorization) untuk menemukan tema/topik yang paling sering muncul pada review dengan `sentiment_label = negative`. Seluruh kategori digabung (tidak dipecah per kategori) karena jumlah review negatif relatif sedikit (~800 dari 65.543 total).

In [ ]:
negative_reviews = df[df['sentiment_label'] == 'negative']['clean_review']
print("Jumlah review negative:", len(negative_reviews))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

stopwords_id = [
    'di','ga','nya','bisa','ini','udah','tapi','aja','dan','ada','yang','yg',
    'tdk','sdh','saya','itu','ke','dari','dengan','untuk','pada','juga','akan',
    'karena','sangat','sekali','banget','ya','jadi','dgn','nggak','gak','tidak',
    'kalo','kalau','sih','deh','dong','kok','loh','lah','pun','atau','saja',
    'sama','dapat','harus','oleh','lagi','masih','sekarang','belum','sudah'
]

tfidf_neg = TfidfVectorizer(max_features=1000, max_df=0.9, min_df=2, stop_words=stopwords_id)
X_neg_tfidf = tfidf_neg.fit_transform(negative_reviews)

print("Bentuk data setelah TF-IDF:", X_neg_tfidf.shape)

In [ ]:
from sklearn.decomposition import NMF

n_topics = 5
nmf_model = NMF(n_components=n_topics, random_state=42, max_iter=500)
nmf_model.fit(X_neg_tfidf)

feature_names = tfidf_neg.get_feature_names_out()

for topic_idx, topic in enumerate(nmf_model.components_):
    top_words_idx = topic.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_words_idx]
    print(f"Topik {topic_idx + 1}: {', '.join(top_words)}")

**Interpretasi topik** (sesuaikan penamaan berdasarkan kata kunci yang muncul di data Anda):
1. Barang Rusak/Cacat Saat Pengiriman
2. Pengiriman Lambat & Kekecewaan
3. Produk Tidak Sesuai Deskripsi/Gambar
4. Kualitas Bahan/Material Kurang Baik
5. Produk Pecah/Busuk (umumnya kategori Makanan & Minuman)

## 6. Menghubungkan Topik Keluhan dengan Kategori Produk

In [ ]:
topic_results = nmf_model.transform(X_neg_tfidf)
dominant_topic = topic_results.argmax(axis=1) + 1

neg_df = df[df['sentiment_label'] == 'negative'].copy()
neg_df['dominant_topic'] = dominant_topic

crosstab = pd.crosstab(neg_df['product_category'], neg_df['dominant_topic'])
crosstab

In [ ]:
# Uji signifikansi statistik: apakah hubungan kategori-topik ini bukan kebetulan?
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(crosstab)
print(f"Chi-square: {chi2:.2f}, p-value: {p_value:.5f}")
print("Signifikan (p < 0.05)?" , "Ya" if p_value < 0.05 else "Tidak")

In [ ]:
# Visualisasi heatmap
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.heatmap(crosstab, annot=True, fmt='d', cmap='Reds')
plt.title('Distribusi Topik Keluhan per Kategori Produk')
plt.xlabel('Topik')
plt.ylabel('Kategori Produk')
plt.tight_layout()
plt.show()

## 7. Kesimpulan & Rekomendasi Bisnis

**Ringkasan temuan:**
- Model klasifikasi sentimen (SVM) berhasil dibangun langsung dari teks review, dengan penanganan khusus untuk data yang sangat *imbalanced*.
- Topic modeling menemukan 5 tema keluhan utama dari review negatif.
- Masalah pengiriman lambat bersifat merata di hampir semua kategori produk (bukan spesifik satu kategori).
- Masalah produk pecah/busuk sangat terkonsentrasi di kategori Makanan & Minuman.
- Masalah kualitas bahan/material lebih menonjol di kategori Olahraga dibanding kategori lain.

**Rekomendasi bisnis:**
1. Prioritaskan perbaikan logistik pengiriman secara lintas kategori.
2. Terapkan SOP kemasan khusus (anti pecah, tahan suhu) untuk kategori Makanan & Minuman.
3. Tingkatkan quality control material/bahan untuk kategori Olahraga.

**Keterbatasan:**
- Jumlah review negatif relatif kecil (798 dari 65.543), sehingga topic modeling tidak dipecah per kategori untuk menjaga keandalan hasil.
- `review_date` hanya memiliki granularitas harian, sehingga analisis tren waktu yang lebih presisi (mis. per jam) tidak dapat dilakukan.
- Rendahnya proporsi review negatif konsisten dengan *selection bias* yang umum terjadi pada platform review, di mana pelanggan puas cenderung lebih sering menulis review dibanding pelanggan kecewa.